# 10 — LLM Cluster Labelling + Consolidation

Processes the step-9 table in batches, asking the LLM to propose cluster names per
batch, then sends **all** resulting cluster names back to the LLM in a single
consolidation call to produce a final set of archetypes.

**Settings** (top of code cell):

- `BATCH_SIZE` — respondents per batch API call (default 1,000)
- `N_BATCH_CLUSTERS` — cluster names to request per batch (default 10)
- `N_FINAL_CLUSTERS` — final archetypes after consolidation (default 10)

**Outputs**:

- `data/10_cluster_LLM/k_llm_clusters.pkl` — per-respondent batch labels + final archetype
- `data/10_cluster_LLM/k_llm_batches/batch_NNNN.json` — per-batch checkpoints (auto-resume)
- `data/10_cluster_LLM/k_consolidation_mapping.json` — consolidation result (cached)
- `api/data/clusters/national_llm_clusters.csv` — pre-aggregated API summary

Requires `OPENAI_API_KEY` in environment or `.env` file in repo root.


In [ ]:
# ── Imports & config ─────────────────────────────────────────────────────────
import sys, os, json, time
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_PKL    = Path(f"../{DATA_FOLDER}/9_cluster_values_k_means/k_with_values_clusters.pkl")
OUT_DIR      = Path(f"../{DATA_FOLDER}/10_cluster_LLM")
BATCH_DIR    = OUT_DIR / "k_llm_batches"
OUT_PKL      = OUT_DIR / "k_llm_clusters.pkl"
MAPPING_JSON = OUT_DIR / "k_consolidation_mapping.json"

for d in (OUT_DIR, BATCH_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Settings ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 500    # respondents per LLM API call
MAX_BATCHES      = 3      # number of API calls to make (None = all batches)
N_BATCH_CLUSTERS = 10     # cluster names to request per batch
N_FINAL_CLUSTERS = 10     # final archetypes after consolidation
MODEL            = "gpt-4o-mini"
RETRY_DELAY_SECS = 10

if not INPUT_PKL.exists():
    raise FileNotFoundError(f"{INPUT_PKL} — run step 9 first.")

df_source = pd.read_pickle(INPUT_PKL)
df_source["pidp"] = pd.to_numeric(df_source["pidp"], errors="coerce").astype("int64")

# ── Profile length diagnostics ────────────────────────────────────────────────
profile_lens = df_source["nl_profile"].dropna().str.len()
print(f"Loaded {len(df_source):,} respondents  ({-(-len(df_source)//BATCH_SIZE)} total batches of {BATCH_SIZE})")
print(f"\nnl_profile length (chars):")
print(f"  min={profile_lens.min():.0f}  median={profile_lens.median():.0f}  "
      f"mean={profile_lens.mean():.0f}  max={profile_lens.max():.0f}")
print(f"\nSample profile (first respondent):\n  {df_source['nl_profile'].iloc[0]}")

# Rough token estimate for a full batch prompt (chars / 4 ≈ tokens)
sample_chars = profile_lens.mean() * BATCH_SIZE
rough_tokens = int(sample_chars / 4)
print(f"\nEstimated prompt size for one batch of {BATCH_SIZE} profiles:")
print(f"  ~{sample_chars:,.0f} chars  ≈  {rough_tokens:,} tokens")
if rough_tokens > 100_000:
    print("  ⚠️  Very large prompt — consider reducing BATCH_SIZE")

n_total_batches = -(-len(df_source) // BATCH_SIZE)
n_run = min(MAX_BATCHES, n_total_batches) if MAX_BATCHES else n_total_batches
print(f"\nWill make {n_run} API call(s), each with up to {BATCH_SIZE} profiles\n")


# ═══════════════════════════════════════════════════════════════════════════════
# ── Batch LLM labelling ───────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

_BATCH_SYSTEM = (
    "You are a social researcher analysing UK household survey respondents. "
    "Read the profiles below and group them into exactly {n} meaningful, "
    "human-readable cluster names that capture distinct social or demographic "
    "patterns. Be specific — 'Young Urban Renters' beats 'Group A'."
).format(n=N_BATCH_CLUSTERS)


def _batch_prompt(profiles: list[str]) -> str:
    numbered = "\n".join(f"{i}. {p}" for i, p in enumerate(profiles))
    return (
        f"Below are {len(profiles)} respondent profiles (numbered 0–{len(profiles)-1}).\n\n"
        f"{numbered}\n\n"
        f"Return ONLY a JSON object with two keys:\n"
        f'  "clusters": array of exactly {N_BATCH_CLUSTERS} cluster name strings\n'
        f'  "assignments": array of exactly {len(profiles)} integers (0–{N_BATCH_CLUSTERS-1}), '
        f"one per profile in the same order.\n"
        f"No explanation, no markdown."
    )


def _call_batch_llm(profiles: list[str], batch_id: int) -> dict:
    prompt = _batch_prompt(profiles)
    prompt_chars = len(prompt)
    print(f"    prompt: {prompt_chars:,} chars ≈ {prompt_chars//4:,} tokens", flush=True)
    for attempt in range(3):
        try:
            t_req = time.time()
            resp  = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": _BATCH_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
                temperature=0.3,
            )
            t_resp = time.time() - t_req
            usage  = resp.usage
            print(f"    response: {t_resp:.1f}s  |  "
                  f"in={usage.prompt_tokens:,}  out={usage.completion_tokens:,} tokens", flush=True)
            parsed = json.loads(resp.choices[0].message.content)
            if len(parsed["clusters"]) != N_BATCH_CLUSTERS:
                raise ValueError(f"Expected {N_BATCH_CLUSTERS} clusters, got {len(parsed['clusters'])}")
            if len(parsed["assignments"]) != len(profiles):
                raise ValueError(f"Expected {len(profiles)} assignments, got {len(parsed['assignments'])}")
            return parsed
        except Exception as exc:
            print(f"    attempt {attempt+1} failed: {exc}", flush=True)
            if attempt < 2:
                print(f"    retrying in {RETRY_DELAY_SECS}s …", flush=True)
                time.sleep(RETRY_DELAY_SECS)
    raise RuntimeError(f"Batch {batch_id} failed after 3 attempts.")


rows    = df_source[["pidp", "nl_profile"]].reset_index(drop=True)
results = []

for batch_id in range(n_run):
    checkpoint = BATCH_DIR / f"batch_{batch_id:04d}.json"
    start      = batch_id * BATCH_SIZE
    end        = min(start + BATCH_SIZE, len(rows))
    batch_rows = rows.iloc[start:end]
    profiles   = batch_rows["nl_profile"].tolist()

    t_batch = time.time()
    if checkpoint.exists():
        with checkpoint.open() as f:
            parsed = json.load(f)
        print(f"API call {batch_id+1}/{n_run} — loaded from checkpoint ({len(profiles)} profiles)")
    else:
        print(f"API call {batch_id+1}/{n_run} — sending {len(profiles)} profiles to {MODEL} …", flush=True)
        parsed = _call_batch_llm(profiles, batch_id)
        with checkpoint.open("w") as f:
            json.dump(parsed, f)
        print(f"    checkpoint saved → {checkpoint.name}", flush=True)

    print(f"    batch wall time: {time.time()-t_batch:.1f}s", flush=True)

    cluster_names = parsed["clusters"]
    for idx, (_, row) in enumerate(batch_rows.iterrows()):
        ci = int(parsed["assignments"][idx])
        results.append({
            "pidp":             int(row["pidp"]),
            "nl_profile":       row["nl_profile"],
            "llm_cluster_name": cluster_names[min(ci, len(cluster_names) - 1)],
            "llm_batch_id":     batch_id,
        })

df = pd.DataFrame(results)
input_names = sorted(df["llm_cluster_name"].dropna().unique().tolist())
print(f"\nBatch labelling done: {len(df):,} respondents, {len(input_names)} distinct cluster names")
print("Cluster names found:")
for name in input_names:
    n = (df["llm_cluster_name"] == name).sum()
    print(f"  {n:>5,}  {name}")


# ═══════════════════════════════════════════════════════════════════════════════
# ── Consolidation ─────────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

_CONSOL_SYSTEM = (
    "You are a social researcher. Given a list of cluster names from a batch "
    "analysis of UK survey respondents, consolidate them into exactly {n} final "
    "archetypes that best capture distinct social and demographic patterns. "
    "Be vivid and specific."
).format(n=N_FINAL_CLUSTERS)


def _consol_prompt(names: list[str]) -> str:
    numbered = "\n".join(f"{i}. {name}" for i, name in enumerate(names))
    return (
        f"Below are {len(names)} cluster names (numbered 0–{len(names)-1}) "
        f"from a batch LLM analysis of UK survey respondents.\n\n"
        f"{numbered}\n\n"
        f"Consolidate into exactly {N_FINAL_CLUSTERS} final archetypes.\n"
        f"Return ONLY a JSON object with two keys:\n"
        f'  "archetypes": array of exactly {N_FINAL_CLUSTERS} archetype name strings\n'
        f'  "mapping": array of exactly {len(names)} integers (0–{N_FINAL_CLUSTERS-1}), '
        f"one per input name in the same order.\n"
        f"No explanation, no markdown."
    )


def _call_consol_llm(names: list[str]) -> dict:
    for attempt in range(3):
        try:
            t_req = time.time()
            resp  = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": _CONSOL_SYSTEM},
                    {"role": "user",   "content": _consol_prompt(names)},
                ],
                temperature=0.3,
            )
            usage = resp.usage
            print(f"  response: {time.time()-t_req:.1f}s  |  "
                  f"in={usage.prompt_tokens:,}  out={usage.completion_tokens:,} tokens", flush=True)
            parsed = json.loads(resp.choices[0].message.content)
            if len(parsed["archetypes"]) != N_FINAL_CLUSTERS:
                raise ValueError(f"Expected {N_FINAL_CLUSTERS} archetypes, got {len(parsed['archetypes'])}")
            if len(parsed["mapping"]) != len(names):
                raise ValueError(f"Expected {len(names)} mapping entries, got {len(parsed['mapping'])}")
            return parsed
        except Exception as exc:
            print(f"  attempt {attempt+1} failed: {exc}", flush=True)
            if attempt < 2:
                print(f"  retrying in {RETRY_DELAY_SECS}s …", flush=True)
                time.sleep(RETRY_DELAY_SECS)
    raise RuntimeError("Consolidation failed after 3 attempts.")


if MAPPING_JSON.exists():
    with MAPPING_JSON.open() as f:
        consol = json.load(f)
    print(f"\nLoaded cached consolidation from {MAPPING_JSON.name}")
else:
    print(f"\nSending {len(input_names)} cluster names to {MODEL} for consolidation …", flush=True)
    t0     = time.time()
    consol = _call_consol_llm(input_names)
    print(f"Consolidation done ({time.time()-t0:.1f}s total)")
    with MAPPING_JSON.open("w") as f:
        json.dump(consol, f, indent=2)
    print(f"Saved → {MAPPING_JSON}")

final_archetypes = consol["archetypes"]
name_to_final = {
    input_names[i]: final_archetypes[idx]
    for i, idx in enumerate(consol["mapping"])
    if idx < len(final_archetypes)
}

df["final_cluster_name"] = df["llm_cluster_name"].map(name_to_final)
n_unmapped = df["final_cluster_name"].isna().sum()
if n_unmapped:
    print(f"WARNING: {n_unmapped} respondents unmapped — check consolidation response")

df.to_pickle(OUT_PKL)
df.to_csv(OUT_PKL.with_suffix(".csv"), index=False)
print(f"\nSaved {len(df):,} rows → {OUT_PKL}")

print(f"\n{N_FINAL_CLUSTERS} final archetypes:")
for arch in final_archetypes:
    print(f"  • {arch}")
print(f"\nFinal cluster distribution:")
print(df["final_cluster_name"].value_counts().to_string())


# ── Export API summary CSV ────────────────────────────────────────────────────
# make_cluster_summary expects a numeric cluster column; map archetype names to IDs
from data_pipeline.config_cluster import WAVE as _WAVE
import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)

df9 = pd.read_pickle(INPUT_PKL)
df9["pidp"] = pd.to_numeric(df9["pidp"], errors="coerce").astype("int64")
df9 = df9.merge(df[["pidp", "final_cluster_name"]], on="pidp", how="left")

# Create a stable numeric ID for each archetype (sorted for reproducibility)
arch_sorted = sorted(df9["final_cluster_name"].dropna().unique())
arch_to_id  = {name: i for i, name in enumerate(arch_sorted)}
df9["_cluster_id"] = df9["final_cluster_name"].map(arch_to_id)

_summary = _cs.make_cluster_summary(df9, "_cluster_id", wave=_WAVE)

# Replace the generic "Cluster N" label with the actual archetype name
id_to_arch = {v: k for k, v in arch_to_id.items()}
_summary["tribe_label"] = _summary["cluster_id"].map(id_to_arch)

_api_dir = Path("../api/data/clusters")
_api_dir.mkdir(parents=True, exist_ok=True)
_summary.to_csv(_api_dir / "national_llm_clusters.csv", index=False)
print(f"\nAPI summary → {_api_dir / 'national_llm_clusters.csv'}")
print(_summary[["cluster_id", "tribe_label", "n_respondents"]].to_string(index=False))


Loaded 27,330 respondents

nl_profile length (chars):
  min=269  median=358  mean=357  max=427

Sample profile (first respondent):
  A 57 year old White: British/English/Scottish/Welsh/N. Irish female, whose highest qualification is gcse etc, Their employment status is unemployed, their marital status is married/civil partner, their housing tenure is rented private unfurnished, and they live in a couple both under pensionable age, no children household. They rate their general health as fair.

Estimated prompt size for 500 profiles:
  ~178,643 chars  ≈  44,660 tokens

Batches of 500: running 3 of 55 total

Batch 1/3 — 500 profiles …
    prompt: 183,285 chars ≈ 45,821 tokens
